# Step 03 of 04 - U-Net++ segmentation model

Train a U-Net++ with timm-efficientnet-b3 backbone to segment tampered pixels in
date-region crops. Input is a 4-channel tensor (RGB + a Laplacian forensic
channel). The loss combines weighted BCE and Dice to handle the very small
positive masks.

In [ ]:
"""
U-Net training for receipt date manipulation segmentation.
Architecture: U-Net + EfficientNet-B3 encoder, BCE+Dice loss with positive weighting.
"""
from __future__ import annotations

import json
import random
from datetime import datetime
from pathlib import Path
from typing import Dict, Iterable, Tuple

import albumentations as A
import cv2
import numpy as np
import segmentation_models_pytorch as smp
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.cuda.amp import GradScaler, autocast
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import DataLoader, Dataset

## Configuration

All hyperparameters and paths. `ML_DIR` resolves to `backend/ml`.

- **`DATASET_ROOT`** / **`OUTPUT_ROOT`** - dataset from Step 03 in, run artifacts out.
- **Model** - U-Net++ arch, EfficientNet-B3 encoder, 4 input channels, 1 output class.
- **Input size** - 64×512 (the date-crop aspect ratio).
- **Training** - batch size, epochs, LR, weight decay, early-stopping patience, and
  the BCE/Dice/positive-weight loss balance.
- **Metrics/threshold** - decision threshold and the pixel count above which an
  image is called "fraud".
- **Normalization** - ImageNet stats for RGB, the Laplacian channel kept in 0..1.

In [ ]:
# Notebook runs from backend/ml/notebooks, so the ml dir is the parent of cwd
ML_DIR = Path.cwd().parent

# ============================================================
# Config
# ============================================================
DATASET_ROOT = ML_DIR / "data/stage_02/dataset"
OUTPUT_ROOT = ML_DIR / "models/stage_02"

# Model
MODEL_ARCH = "unetplusplus"
ENCODER = "timm-efficientnet-b3"
ENCODER_WEIGHTS = "imagenet"
IN_CHANNELS = 4
CLASSES = 1

# Input: receipt date-region crops
INPUT_HEIGHT = 64
INPUT_WIDTH = 512

# Training
BATCH_SIZE = 16
EPOCHS = 200
LR = 5e-4
WEIGHT_DECAY = 1e-4
PATIENCE = 40
POS_WEIGHT = 2.0
DICE_WEIGHT = 0.5
BCE_WEIGHT = 1.0

# Metrics / decision threshold
DEFAULT_THRESHOLD = 0.5
IMAGE_POS_PIXEL_THRESHOLD = 10  # image is "fraud" if >= this many predicted positive pixels
MONITOR_METRIC = "f1"           # better than pixel accuracy for tiny masks

# Reproducibility / loader
SEED = 42
NUM_WORKERS = 4

# Normalization: RGB uses ImageNet stats; Laplacian is scaled to 0..1.
RGB_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
RGB_STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)
LAP_MEAN = 0.0
LAP_STD = 1.0

## Reproducibility helpers

`seed_everything` seeds Python/NumPy/Torch; `seed_worker` seeds each DataLoader
worker for deterministic augmentation.

In [ ]:
# ============================================================
# Reproducibility helpers
# ============================================================

def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True


def seed_worker(worker_id: int) -> None:
    worker_seed = (SEED + worker_id) % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)

## Forensic preprocessing

Builds the 4th input channel. `compute_laplacian_channel` produces a Laplacian
magnitude map (a tampering prior), `normalize_rgb` applies ImageNet normalization,
and `to_4ch_tensor` stacks them into a `(4, H, W)` tensor - crucially computing the
Laplacian after augmentation so it matches the image the model actually sees.

In [ ]:
# ============================================================
# Forensic preprocessing
# ============================================================

def compute_laplacian_channel(rgb_uint8: np.ndarray) -> np.ndarray:
    """Return a single-channel Laplacian magnitude image in 0..1 float32."""
    gray = cv2.cvtColor(rgb_uint8, cv2.COLOR_RGB2GRAY)
    lap = cv2.Laplacian(gray, cv2.CV_32F, ksize=3)
    lap = np.abs(lap)
    lap = np.clip(lap, 0, 255).astype(np.float32) / 255.0
    lap = (lap - LAP_MEAN) / LAP_STD
    return lap[..., None]


def normalize_rgb(rgb_uint8: np.ndarray) -> np.ndarray:
    rgb = rgb_uint8.astype(np.float32) / 255.0
    rgb = (rgb - RGB_MEAN) / RGB_STD
    return rgb


def to_4ch_tensor(rgb_uint8: np.ndarray) -> torch.Tensor:
    """
    Convert augmented RGB uint8 image to normalized 4-channel tensor.

    Important: the Laplacian is computed AFTER augmentation, so it matches the
    actual image seen by the model.
    """
    rgb_norm = normalize_rgb(rgb_uint8)
    lap_norm = compute_laplacian_channel(rgb_uint8)
    stacked = np.concatenate([rgb_norm, lap_norm], axis=-1)  # H, W, 4
    stacked = np.ascontiguousarray(stacked.transpose(2, 0, 1))  # 4, H, W
    return torch.from_numpy(stacked).float()

## Dataset

`ReceiptManipulationDataset` loads each image and its matching mask (treating a
missing mask as a clean, all-zero target), applies the Albumentations transform to
RGB+mask, then converts to the 4-channel tensor and a binary mask tensor.

In [ ]:
# ============================================================
# Dataset
# ============================================================

class ReceiptManipulationDataset(Dataset):
    def __init__(self, images_dir: Path, masks_dir: Path, transform: A.Compose | None = None):
        self.images_dir = Path(images_dir)
        self.masks_dir = Path(masks_dir)
        self.transform = transform

        if not self.images_dir.exists():
            raise FileNotFoundError(f"Images directory not found: {self.images_dir}")

        self.image_paths = sorted(
            p for p in self.images_dir.iterdir()
            if p.suffix.lower() in {".png", ".jpg", ".jpeg", ".webp", ".bmp", ".tif", ".tiff"}
        )

        if len(self.image_paths) == 0:
            raise RuntimeError(f"No images found in: {self.images_dir}")

    def __len__(self) -> int:
        return len(self.image_paths)

    def _find_mask_path(self, img_path: Path) -> Path | None:
        # First try exact same filename, then same stem with common mask extensions.
        exact = self.masks_dir / img_path.name
        if exact.exists():
            return exact

        for suffix in (".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff", ".webp"):
            candidate = self.masks_dir / f"{img_path.stem}{suffix}"
            if candidate.exists():
                return candidate

        return None

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, torch.Tensor]:
        img_path = self.image_paths[idx]
        mask_path = self._find_mask_path(img_path)

        bgr = cv2.imread(str(img_path), cv2.IMREAD_COLOR)
        if bgr is None:
            raise RuntimeError(f"Could not read image: {img_path}")
        rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)

        if mask_path is None:
            # Missing mask means clean receipt/date crop.
            mask = np.zeros(rgb.shape[:2], dtype=np.uint8)
        else:
            mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
            if mask is None:
                raise RuntimeError(f"Could not read mask: {mask_path}")

        # Convert any 0/255 mask to 0/1. Keep as uint8 before augmentation.
        mask = (mask > 0).astype(np.uint8)

        if self.transform is not None:
            transformed = self.transform(image=rgb, mask=mask)
            rgb = transformed["image"]
            mask = transformed["mask"]

        # Some augmentations may return non-uint8; convert safely before Laplacian.
        if rgb.dtype != np.uint8:
            rgb = np.clip(rgb, 0, 255).astype(np.uint8)

        image_tensor = to_4ch_tensor(rgb)

        mask = (mask > 0).astype(np.float32)
        mask_tensor = torch.from_numpy(mask).unsqueeze(0).float()  # 1, H, W

        return image_tensor, mask_tensor

## Transforms

Augmentation pipelines with version-compatibility shims for Albumentations v1/v2.
Training applies brightness/contrast, hue/saturation, gamma, JPEG compression,
blur, noise, and small geometric jitter validation only resizes. Augmentation
acts on RGB+mask only - the Laplacian is recomputed afterward.

In [ ]:
# ============================================================
# Transforms
# ============================================================

def resize_transform() -> A.BasicTransform:
    """Albumentations v1/v2 compatible resize."""
    try:
        return A.Resize(
            INPUT_HEIGHT,
            INPUT_WIDTH,
            interpolation=cv2.INTER_LINEAR,
            mask_interpolation=cv2.INTER_NEAREST,
        )
    except TypeError:
        # Older Albumentations versions do not expose mask_interpolation.
        return A.Resize(INPUT_HEIGHT, INPUT_WIDTH, interpolation=cv2.INTER_LINEAR)


def image_compression_transform(quality_low: int, quality_high: int) -> A.BasicTransform:
    """Albumentations v1/v2 compatible JPEG compression."""
    try:
        return A.ImageCompression(quality_range=(quality_low, quality_high), p=1.0)
    except TypeError:
        return A.ImageCompression(quality_lower=quality_low, quality_upper=quality_high, p=1.0)


def small_geometric_transform() -> A.BasicTransform:
    """Small rotation/shift/scale with conservative borders."""
    try:
        return A.ShiftScaleRotate(
            shift_limit=0.05,
            scale_limit=0.10,
            rotate_limit=3,
            border_mode=cv2.BORDER_REPLICATE,
            value=None,
            mask_value=0,
            p=0.40,
        )
    except TypeError:
        return A.ShiftScaleRotate(
            shift_limit=0.05,
            scale_limit=0.10,
            rotate_limit=3,
            border_mode=cv2.BORDER_REPLICATE,
            p=0.40,
        )

def get_train_transform() -> A.Compose:
    """
    Augment RGB + mask only.

    The Laplacian channel is not included here. Instead it is computed after
    the augmentation so it remains a valid forensic prior.
    """
    return A.Compose([
        resize_transform(),

        A.RandomBrightnessContrast(brightness_limit=0.20, contrast_limit=0.20, p=0.60),
        A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=20, val_shift_limit=15, p=0.50),
        A.RandomGamma(gamma_limit=(80, 120), p=0.30),

        A.OneOf([
            image_compression_transform(40, 70),
            image_compression_transform(85, 100),
        ], p=0.60),

        A.OneOf([
            A.GaussianBlur(blur_limit=(3, 5), p=1.0),
            A.MotionBlur(blur_limit=5, p=1.0),
            A.MedianBlur(blur_limit=3, p=1.0),
        ], p=0.20),

        A.OneOf([
            A.GaussNoise(p=1.0),
            A.ISONoise(p=1.0),
        ], p=0.30),

        small_geometric_transform(),
    ])


def get_val_transform() -> A.Compose:
    return A.Compose([
        resize_transform(),
    ])

## Loss

`CombinedLoss` sums positive-weighted BCE (computed from logits) and Dice loss,
which together cope with the heavy class imbalance of tiny tamper masks.

In [ ]:
# ============================================================
# Loss
# ============================================================

class CombinedLoss(nn.Module):
    def __init__(self, pos_weight: float = 2.0, bce_weight: float = 1.0, dice_weight: float = 1.0):
        super().__init__()
        self.register_buffer("pos_weight", torch.tensor([pos_weight], dtype=torch.float32))
        self.dice = smp.losses.DiceLoss(mode="binary", from_logits=True)
        self.bce_weight = bce_weight
        self.dice_weight = dice_weight

    def forward(self, logits: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
        target = target.float()

        bce_loss = F.binary_cross_entropy_with_logits(
            logits,
            target,
            pos_weight=self.pos_weight,
        )
        dice_loss = self.dice(logits, target)

        return self.bce_weight * bce_loss + self.dice_weight * dice_loss

## Metrics

Pixel- and image-level metric accumulation. `update_metric_counts` tallies
TP/FP/FN/TN and per-image fraud decisions; `finalize_metrics` turns those into
loss, IoU, precision/recall/F1, pixel accuracy, and image-level FP rate / recall.

In [ ]:
# ============================================================
# Metrics
# ============================================================

def empty_metric_counts() -> Dict[str, float]:
    return {
        "loss_sum": 0.0,
        "num_batches": 0,
        "tp": 0.0,
        "fp": 0.0,
        "fn": 0.0,
        "tn": 0.0,
        "total_pixels": 0.0,
        "clean_images": 0.0,
        "forged_images": 0.0,
        "clean_false_positive_images": 0.0,
        "forged_detected_images": 0.0,
    }


def update_metric_counts(
    counts: Dict[str, float],
    logits: torch.Tensor,
    target: torch.Tensor,
    loss: torch.Tensor | None = None,
    threshold: float = DEFAULT_THRESHOLD,
    image_pos_pixel_threshold: int = IMAGE_POS_PIXEL_THRESHOLD,
) -> None:
    if loss is not None:
        counts["loss_sum"] += float(loss.item())
        counts["num_batches"] += 1

    probs = torch.sigmoid(logits)
    pred = probs > threshold
    target_bool = target > 0.5

    tp = (pred & target_bool).sum().item()
    fp = (pred & ~target_bool).sum().item()
    fn = (~pred & target_bool).sum().item()
    tn = (~pred & ~target_bool).sum().item()

    counts["tp"] += tp
    counts["fp"] += fp
    counts["fn"] += fn
    counts["tn"] += tn
    counts["total_pixels"] += target.numel()

    # Image-level fraud/clean decision. Useful because clean all-zero masks can
    # make pixel accuracy misleadingly high.
    pred_pixels_per_image = pred.flatten(1).sum(dim=1)
    target_pixels_per_image = target_bool.flatten(1).sum(dim=1)

    pred_fraud = pred_pixels_per_image >= image_pos_pixel_threshold
    target_fraud = target_pixels_per_image > 0
    target_clean = ~target_fraud

    counts["clean_images"] += target_clean.sum().item()
    counts["forged_images"] += target_fraud.sum().item()
    counts["clean_false_positive_images"] += (pred_fraud & target_clean).sum().item()
    counts["forged_detected_images"] += (pred_fraud & target_fraud).sum().item()


def finalize_metrics(counts: Dict[str, float], eps: float = 1e-7) -> Dict[str, float]:
    tp = counts["tp"]
    fp = counts["fp"]
    fn = counts["fn"]
    tn = counts["tn"]

    loss = counts["loss_sum"] / max(counts["num_batches"], 1)
    pixel_acc = (tp + tn) / max(counts["total_pixels"], 1)

    precision = tp / (tp + fp + eps)
    recall = tp / (tp + fn + eps)
    f1 = 2 * precision * recall / (precision + recall + eps)
    iou = tp / (tp + fp + fn + eps)

    clean_images = counts["clean_images"]
    forged_images = counts["forged_images"]
    image_fp_rate = counts["clean_false_positive_images"] / max(clean_images, 1)
    image_recall = counts["forged_detected_images"] / max(forged_images, 1)

    return {
        "loss": loss,
        "iou": iou,
        "pixel_acc": pixel_acc,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "image_fp_rate": image_fp_rate,
        "image_recall": image_recall,
    }

## Model

`build_model` constructs the U-Net++ (or plain U-Net) with the configured encoder
and SCSE decoder attention, with fallbacks for differing SMP versions.

In [ ]:
# ============================================================
# Model
# ============================================================

def build_model() -> nn.Module:
    common_kwargs = dict(
        encoder_name=ENCODER,
        encoder_weights=ENCODER_WEIGHTS,
        in_channels=IN_CHANNELS,
        classes=CLASSES,
        activation=None,
        decoder_attention_type="scse",
    )

    # Added some extra generalization as SMP versions was differing
    # Some do accept decoder_interpolation, some do not.
    if MODEL_ARCH.lower() in {"unetplusplus", "unet++", "unet_plus_plus"}:
        try:
            return smp.UnetPlusPlus(**common_kwargs, decoder_interpolation="bilinear")
        except TypeError:
            return smp.UnetPlusPlus(**common_kwargs)

    if MODEL_ARCH.lower() == "unet":
        try:
            return smp.Unet(**common_kwargs, decoder_interpolation="bilinear")
        except TypeError:
            return smp.Unet(**common_kwargs)

    raise ValueError(f"Unsupported MODEL_ARCH: {MODEL_ARCH}")

## Training & validation loops

`train_one_epoch` runs a mixed-precision training pass with gradient scaling;
`validate` runs evaluation; `evaluate_thresholds` sweeps decision thresholds to
find the one that maximizes the monitored metric on validation data.

In [ ]:
# ============================================================
# Training / validation loops
# ============================================================

def train_one_epoch(
    model: nn.Module,
    loader: DataLoader,
    criterion: nn.Module,
    optimizer: torch.optim.Optimizer,
    scaler: GradScaler,
    device: torch.device,
    threshold: float = DEFAULT_THRESHOLD,
) -> Tuple[float, Dict[str, float]]:
    model.train()
    counts = empty_metric_counts()

    for imgs, masks in loader:
        imgs = imgs.to(device, non_blocking=True)
        masks = masks.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with autocast(enabled=device.type == "cuda"):
            logits = model(imgs)
            loss = criterion(logits, masks)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        with torch.no_grad():
            update_metric_counts(counts, logits.detach(), masks, loss, threshold=threshold)

    metrics = finalize_metrics(counts)
    return metrics["loss"], metrics


@torch.no_grad()
def validate(
    model: nn.Module,
    loader: DataLoader,
    criterion: nn.Module,
    device: torch.device,
    threshold: float = DEFAULT_THRESHOLD,
) -> Tuple[float, Dict[str, float]]:
    model.eval()
    counts = empty_metric_counts()

    for imgs, masks in loader:
        imgs = imgs.to(device, non_blocking=True)
        masks = masks.to(device, non_blocking=True)

        with autocast(enabled=device.type == "cuda"):
            logits = model(imgs)
            loss = criterion(logits, masks)

        update_metric_counts(counts, logits, masks, loss, threshold=threshold)

    metrics = finalize_metrics(counts)
    return metrics["loss"], metrics


@torch.no_grad()
def evaluate_thresholds(
    model: nn.Module,
    loader: DataLoader,
    criterion: nn.Module,
    device: torch.device,
    thresholds: Iterable[float] = np.arange(0.10, 0.91, 0.05),
    metric_name: str = "f1",
) -> Tuple[float, Dict[str, float]]:
    best_threshold = DEFAULT_THRESHOLD
    best_metrics: Dict[str, float] | None = None
    best_score = -1.0

    for threshold in thresholds:
        _, metrics = validate(model, loader, criterion, device, threshold=float(threshold))
        score = metrics[metric_name]
        if score > best_score:
            best_score = score
            best_threshold = float(threshold)
            best_metrics = metrics

    assert best_metrics is not None
    return best_threshold, best_metrics

## Checkpoint loading

`load_checkpoint` loads a saved checkpoint, with a fallback for older PyTorch
versions lacking the `weights_only` argument.

In [ ]:
# ============================================================
# Checkpoint loading
# ============================================================

def load_checkpoint(path: Path, device: torch.device) -> dict:
    try:
        return torch.load(path, map_location=device, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=device)

## Main routine

The full training program: build datasets/loaders, model, loss, optimizer and
schedule; train with early stopping while checkpointing the best model; then
reload it, pick the best threshold on validation, evaluate on test, and write
`config.json`, `test_results.json`, and `test_results.txt`.

In [ ]:
# ============================================================
# Main
# ============================================================

def main() -> None:
    seed_everything(SEED)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Device: {device}")
    if device.type == "cuda":
        print(f"GPU: {torch.cuda.get_device_name(0)}")

    run_name = f"{MODEL_ARCH}_{ENCODER.replace('/', '_')}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
    run_dir = OUTPUT_ROOT / run_name
    run_dir.mkdir(parents=True, exist_ok=True)
    print(f"Outputs to: {run_dir}")

    train_ds = ReceiptManipulationDataset(
        DATASET_ROOT / "images" / "train",
        DATASET_ROOT / "masks" / "train",
        transform=get_train_transform(),
    )
    val_ds = ReceiptManipulationDataset(
        DATASET_ROOT / "images" / "val",
        DATASET_ROOT / "masks" / "val",
        transform=get_val_transform(),
    )
    test_ds = ReceiptManipulationDataset(
        DATASET_ROOT / "images" / "test",
        DATASET_ROOT / "masks" / "test",
        transform=get_val_transform(),
    )
    print(f"Train: {len(train_ds)}, Val: {len(val_ds)}, Test: {len(test_ds)}")

    generator = torch.Generator()
    generator.manual_seed(SEED)

    train_loader = DataLoader(
        train_ds,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=(device.type == "cuda"),
        worker_init_fn=seed_worker,
        generator=generator,
        drop_last=False,
    )
    val_loader = DataLoader(
        val_ds,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=(device.type == "cuda"),
        drop_last=False,
    )
    test_loader = DataLoader(
        test_ds,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=(device.type == "cuda"),
        drop_last=False,
    )

    model = build_model().to(device)
    n_params = sum(p.numel() for p in model.parameters())
    print(f"Model: {MODEL_ARCH} + {ENCODER}, params: {n_params / 1e6:.2f}M")

    criterion = CombinedLoss(
        pos_weight=POS_WEIGHT,
        bce_weight=BCE_WEIGHT,
        dice_weight=DICE_WEIGHT,
    ).to(device)

    optimizer = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=LR * 0.01)
    scaler = GradScaler(enabled=(device.type == "cuda"))

    config = {
        "model_arch": MODEL_ARCH,
        "encoder": ENCODER,
        "encoder_weights": ENCODER_WEIGHTS,
        "in_channels": IN_CHANNELS,
        "input_height": INPUT_HEIGHT,
        "input_width": INPUT_WIDTH,
        "batch_size": BATCH_SIZE,
        "epochs": EPOCHS,
        "lr": LR,
        "weight_decay": WEIGHT_DECAY,
        "patience": PATIENCE,
        "pos_weight": POS_WEIGHT,
        "dice_weight": DICE_WEIGHT,
        "bce_weight": BCE_WEIGHT,
        "monitor_metric": MONITOR_METRIC,
        "default_threshold": DEFAULT_THRESHOLD,
        "image_pos_pixel_threshold": IMAGE_POS_PIXEL_THRESHOLD,
        "seed": SEED,
    }
    with open(run_dir / "config.json", "w", encoding="utf-8") as f:
        json.dump(config, f, indent=2)

    best_score = -1.0
    best_val_metrics: Dict[str, float] | None = None
    epochs_without_improvement = 0

    print(
        f"\n{'Epoch':<7}{'TrLoss':<10}{'TrIoU':<8}{'TrF1':<8}"
        f"{'VlLoss':<10}{'VlIoU':<8}{'VlF1':<8}{'VlFPimg':<9}{'LR':<10}"
    )
    print("-" * 88)

    for epoch in range(1, EPOCHS + 1):
        train_loss, train_metrics = train_one_epoch(
            model, train_loader, criterion, optimizer, scaler, device, threshold=DEFAULT_THRESHOLD
        )
        val_loss, val_metrics = validate(
            model, val_loader, criterion, device, threshold=DEFAULT_THRESHOLD
        )
        scheduler.step()

        lr_now = optimizer.param_groups[0]["lr"]
        print(
            f"{epoch:<7}{train_loss:<10.4f}{train_metrics['iou']:<8.4f}{train_metrics['f1']:<8.4f}"
            f"{val_loss:<10.4f}{val_metrics['iou']:<8.4f}{val_metrics['f1']:<8.4f}"
            f"{val_metrics['image_fp_rate']:<9.4f}{lr_now:<10.6f}"
        )

        current_score = val_metrics[MONITOR_METRIC]
        if current_score > best_score:
            best_score = current_score
            best_val_metrics = val_metrics
            epochs_without_improvement = 0

            torch.save({
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "scheduler_state_dict": scheduler.state_dict(),
                "best_score": best_score,
                "best_val_metrics": best_val_metrics,
                "config": config,
            }, run_dir / "best.pt")
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= PATIENCE:
                print(f"\nEarly stopping at epoch {epoch} — no {MONITOR_METRIC} improvement for {PATIENCE} epochs.")
                break

    if not (run_dir / "best.pt").exists():
        raise RuntimeError("No checkpoint was saved. Check that training/validation data is valid.")

    print(f"\n{'=' * 60}")
    print(f"Best validation {MONITOR_METRIC}: {best_score:.4f}")
    print("Loading best model...")

    ckpt = load_checkpoint(run_dir / "best.pt", device)
    model.load_state_dict(ckpt["model_state_dict"])

    print("Searching best validation threshold...")
    best_threshold, val_threshold_metrics = evaluate_thresholds(
        model, val_loader, criterion, device, metric_name=MONITOR_METRIC
    )

    print(f"Best threshold by validation {MONITOR_METRIC}: {best_threshold:.2f}")
    print("Evaluating on test set...")
    test_loss, test_metrics = validate(model, test_loader, criterion, device, threshold=best_threshold)

    print("\nVALIDATION METRICS AT BEST THRESHOLD:")
    for k, v in val_threshold_metrics.items():
        print(f"  {k:<15}: {v:.4f}")

    print("\nTEST SET METRICS:")
    for k, v in test_metrics.items():
        print(f"  {k:<15}: {v:.4f}")

    results = {
        "best_epoch": ckpt["epoch"],
        "best_default_threshold_val_metrics": ckpt["best_val_metrics"],
        "best_threshold": best_threshold,
        "val_metrics_at_best_threshold": val_threshold_metrics,
        "test_metrics": test_metrics,
    }

    with open(run_dir / "test_results.json", "w", encoding="utf-8") as f:
        json.dump(results, f, indent=2)

    with open(run_dir / "test_results.txt", "w", encoding="utf-8") as f:
        f.write(f"Best epoch: {ckpt['epoch']}\n")
        f.write(f"Best validation {MONITOR_METRIC} at threshold {DEFAULT_THRESHOLD}: {best_score:.4f}\n")
        f.write(f"Best threshold: {best_threshold:.2f}\n\n")
        f.write("Validation metrics at best threshold:\n")
        for k, v in val_threshold_metrics.items():
            f.write(f"  {k}: {v:.4f}\n")
        f.write("\nTest metrics:\n")
        for k, v in test_metrics.items():
            f.write(f"  {k}: {v:.4f}\n")

    print(f"\nDone. Best model saved to: {run_dir / 'best.pt'}")
    print(f"Results saved to: {run_dir / 'test_results.json'}")

## Run

Launch training and evaluation.

In [ ]:
main()